In [ ]:
# -*- coding: utf-8 -*-
"""
阶段 0-3：数据读取与合并 → 数据清洗 → 特征工程
输出：
  1) 控制台打印每步校验信息
  2) 桌面"代码结果"文件夹导出：
     - 合并原始数据预览 (merged_raw_preview.csv)      # 前100行，仅供核对，不含敏感导出
     - 清洗后数据 (cleaned_data.csv)
     - 特征工程后特征矩阵 (X_features.csv)
     - 标签 (y_labels.csv)
     - 划分信息 (split_info.csv)
  3) 划分好的 X_train/X_test/y_train/y_test 以局部变量形式保留在notebook内存中，
     供后续阶段直接使用。
"""
import os
import pandas as pd
import numpy as np

# ==================== 路径配置 ====================
FILES = [
    r"C:\Users\DELL\Desktop\心理测评\测评结果-2021-1013.xlsx",
    r"C:\Users\DELL\Desktop\心理测评\测评结果-2022-1269.xlsx",
    r"C:\Users\DELL\Desktop\心理测评\测评结果-2023-1276.xlsx",
    r"C:\Users\DELL\Desktop\心理测评\测评结果-2024-1283.xlsx",
    r"C:\Users\DELL\Desktop\心理测评\测评结果-2025-1387.xlsx",
]
OUT_DIR = r"C:\Users\DELL\Desktop\代码结果"
os.makedirs(OUT_DIR, exist_ok=True)

SHEETS = ["严重心理危机", "一般心理问题", "潜在心理困扰", "无心理困扰"]

# ==================== 阶段0：数据读取与合并 ====================
print("=" * 50)
print("阶段0：数据读取与合并")
all_dfs = []
for fp in FILES:
    if not os.path.exists(fp):
        print(f"  ! 文件不存在，跳过：{fp}")
        continue
    # sheet_name=None 读全部sheet，再按目标sheet名筛选，避免sheet名差异报错
    full = pd.read_excel(fp, sheet_name=None)
    for s in SHEETS:
        if s in full:
            all_dfs.append(full[s])
    print(f"  √ 已读取：{os.path.basename(fp)}")

merged = pd.concat(all_dfs, ignore_index=True)
print(f"合并后总行数：{merged.shape[0]}，列数：{merged.shape[1]}")
# 导出前100行预览，方便核对合并是否正常
merged.head(100).to_csv(os.path.join(OUT_DIR, "merged_raw_preview.csv"), index=False, encoding="utf-8-sig")

# ==================== 阶段1：数据清洗 ====================
print("=" * 50)
print("阶段1：数据清洗")

# 一致性得分转数值
merged["一致性得分"] = pd.to_numeric(merged["一致性得分"], errors="coerce")

# 剔除一致性得分<2
before = merged.shape[0]
cleaned = merged[merged["一致性得分"] >= 2].reset_index(drop=True)
after = cleaned.shape[0]
print(f"一致性过滤前：{before} 行")
print(f"一致性过滤后：{after} 行（剔除 {before - after} 行）")

# 构建目标变量
cleaned["自杀意图"] = cleaned["可能问题"].astype(str).str.contains("自杀意图", na=False).astype(int)
pos = cleaned["自杀意图"].sum()
print(f"阳性样本：{pos}，占比：{pos / len(cleaned):.2%}")
print(f"最终有效样本：{len(cleaned)}")

# 导出清洗后数据（去掉可能混入的空行）
cleaned.to_csv(os.path.join(OUT_DIR, "cleaned_data.csv"), index=False, encoding="utf-8-sig")

# ==================== 阶段2：特征工程 ====================
print("=" * 50)
print("阶段2：特征工程")

# 目标变量
y = cleaned["自杀意图"].values

# 剔除无关列
drop_cols = [
    "学历层次", "入学年份", "出生日期", "总分",
    "家庭关系满意度", "学校及专业满意度",
    "可能问题", "测评等级", "提交时间",
    "一致性得分", "总用时(秒)",
    "自杀意图(指标总分)", "自杀意图(指标标准分)",
    "自杀意图",  # 目标列本身
]
# 动态剔除所有 指标总分 列
drop_cols += [c for c in cleaned.columns if "(指标总分)" in c]

cat_cols = ["性别", "民族", "生源地", "是否独生"]

# 数值特征：既不在drop_cols，也不是分类列
num_cols = [c for c in cleaned.columns if c not in drop_cols and c not in cat_cols]

# 去掉"(指标标准分)"后缀
rename_map = {c: c.replace("(指标标准分)", "").strip() for c in num_cols}
cleaned = cleaned.rename(columns=rename_map)
num_cols_new = [rename_map[c] for c in num_cols]

print(f"连续特征数：{len(num_cols_new)}")
print(f"分类特征数：{len(cat_cols)}（独热编码前）")

# 独热编码（drop='first'）
enc_cols = []
for col in cat_cols:
    if cleaned[col].dtype == object:
        cleaned[col] = cleaned[col].astype(str)
enc = pd.get_dummies(cleaned[cat_cols], drop_first=True)
print(f"独热编码后分类特征数：{enc.shape[1]}")

# 拼接特征矩阵
X = pd.concat([cleaned[num_cols_new].reset_index(drop=True),
               enc.reset_index(drop=True)], axis=1)
print(f"特征矩阵 X：{X.shape[0]} 行 × {X.shape[1]} 列")

# 导出特征矩阵与标签
X.to_csv(os.path.join(OUT_DIR, "X_features.csv"), index=False, encoding="utf-8-sig")
pd.Series(y, name="自杀意图").to_csv(os.path.join(OUT_DIR, "y_labels.csv"), index=False, encoding="utf-8-sig")

# ==================== 阶段3：数据集划分 ====================
print("=" * 50)
print("阶段3：分层划分训练集/测试集")
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"训练集：{X_train.shape[0]} 行，阳性 {y_train.sum()}（{y_train.mean():.2%}）")
print(f"测试集：{X_test.shape[0]} 行，阳性 {y_test.sum()}（{y_test.mean():.2%}）")

# 导出划分信息
split_info = pd.DataFrame({
    "集合": ["训练集", "测试集"],
    "样本数": [X_train.shape[0], X_test.shape[0]],
    "阳性数": [int(y_train.sum()), int(y_test.sum())],
    "阳性占比": [f"{y_train.mean():.2%}", f"{y_test.mean():.2%}"],
})
split_info.to_csv(os.path.join(OUT_DIR, "split_info.csv"), index=False, encoding="utf-8-sig")

# 训练/测试集特征与标签也导出，方便你后续与feature矩阵核对
pd.DataFrame(X_train).to_csv(os.path.join(OUT_DIR, "X_train.csv"), index=False, encoding="utf-8-sig")
pd.Series(y_train, name="自杀意图").to_csv(os.path.join(OUT_DIR, "y_train.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame(X_test).to_csv(os.path.join(OUT_DIR, "X_test.csv"), index=False, encoding="utf-8-sig")
pd.Series(y_test, name="自杀意图").to_csv(os.path.join(OUT_DIR, "y_test.csv"), index=False, encoding="utf-8-sig")

print("=" * 50)
print("阶段0-3 完成，输出目录：", OUT_DIR)
print("内存中变量：merged, cleaned, X, y, X_train, X_test, y_train, y_test")

In [1]:
# -*- coding: utf-8 -*-
# Cell 1：环境准备、数据加载、SMOTE-ENC增强
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, chi2
from sklearn.metrics import (roc_auc_score, roc_curve, auc, brier_score_loss,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from imblearn.over_sampling import SMOTENC
import xgboost as xgb
import lightgbm as lgb
import shap
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 300

OUT = r"C:\Users\DELL\Desktop\代码结果"
os.makedirs(OUT, exist_ok=True)

# 加载阶段0-3已保存的数据
X_train = pd.read_csv(os.path.join(OUT, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(OUT, "X_test.csv"))
y_train = pd.read_csv(os.path.join(OUT, "y_train.csv"))["自杀意图"].astype(int).values
y_test  = pd.read_csv(os.path.join(OUT, "y_test.csv"))["自杀意图"].astype(int).values
print(f"训练集 {X_train.shape}，阳性 {y_train.sum()}（{y_train.mean():.2%}）")
print(f"测试集 {X_test.shape}，阳性 {y_test.sum()}（{y_test.mean():.2%}）")

# SMOTE-ENC：仅训练集，平衡到50%
prefix = ("性别_", "民族_", "生源地_", "是否独生_")
cat_idx = [i for i, c in enumerate(X_train.columns) if c.startswith(prefix)]
smote = SMOTENC(categorical_features=cat_idx, random_state=42, sampling_strategy=1.0)
X_bal, y_bal = smote.fit_resample(X_train, y_train)
print(f"增强后训练集 {X_bal.shape}，阳性占比 {y_bal.mean():.2%}")

训练集 (2147, 28)，阳性 369（17.19%）
测试集 (921, 28)，阳性 158（17.16%）
增强后训练集 (3556, 28)，阳性占比 50.00%


In [2]:
# -*- coding: utf-8 -*-
# Cell 2：用已调好的最优参数定义六模型，分别在增强前/增强后训练
def build_models():
    return {
        "逻辑回归": Pipeline([("scaler", StandardScaler()),
            ("model", LogisticRegression(C=0.003, l1_ratio=0.38, penalty="elasticnet",
                                         solver="saga", max_iter=3000, random_state=42,
                                         class_weight={0:1,1:6}))]),
        "决策树": DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=1,
                                         class_weight={0:1,1:3}, splitter="best", random_state=42),
        "随机森林": RandomForestClassifier(n_estimators=300, max_depth=3, max_features="log2",
                                            min_samples_leaf=1, class_weight={0:1,1:3},
                                            random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=300, max_depth=3, max_features="log2",
                                            min_samples_leaf=1, class_weight={0:1,1:2},
                                            random_state=42, n_jobs=-1),
        "XGBoost": xgb.XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3,
                                      min_child_weight=3, scale_pos_weight=3, subsample=0.8,
                                      colsample_bytree=0.8, random_state=42, n_jobs=-1),
        "LightGBM": lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, max_depth=3,
                                        num_leaves=15, min_child_samples=15, scale_pos_weight=4,
                                        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05,
                                        reg_lambda=1.5, random_state=42, n_jobs=-1, verbose=-1),
    }

models = build_models()

# 训练增强前模型 + 记录指标
proba_before, proba_after = {}, {}
for name, m in models.items():
    m.fit(X_train, y_train)
    proba_before[name] = m.predict_proba(X_test)[:, 1]

# 训练增强后模型 + 记录指标
for name, m in models.items():
    m.fit(X_bal, y_bal)
    proba_after[name] = m.predict_proba(X_test)[:, 1]

print("六模型训练完成（增强前+增强后）。")

六模型训练完成（增强前+增强后）。


In [3]:
# -*- coding: utf-8 -*-
# Cell 3：SMOTE-ENC增强前后 召回率/AUC 对比表
rows = []
for name in models:
    yp_b = proba_before[name]
    yp_a = proba_after[name]
    rows.append({
        "模型": name,
        "增强前召回率": round(recall_score(y_test, (yp_b>=0.5).astype(int), zero_division=0), 4),
        "增强前AUC": round(roc_auc_score(y_test, yp_b), 4),
        "增强后召回率": round(recall_score(y_test, (yp_a>=0.5).astype(int), zero_division=0), 4),
        "增强后AUC": round(roc_auc_score(y_test, yp_a), 4),
    })
df_compare = pd.DataFrame(rows)
df_compare.to_csv(os.path.join(OUT, "六模型增强前后对比.csv"), index=False, encoding="utf-8-sig")
print(df_compare.to_string(index=False))

        模型  增强前召回率  增强前AUC  增强后召回率  增强后AUC
      逻辑回归  0.8101  0.8578  0.9810  0.8459
       决策树  0.6392  0.8264  0.8987  0.7905
      随机森林  0.6203  0.8562  0.9177  0.8316
ExtraTrees  0.2785  0.8511  0.9620  0.8354
   XGBoost  0.6835  0.8620  0.8354  0.8518
  LightGBM  0.7152  0.8612  0.8101  0.8446


In [4]:
# -*- coding: utf-8 -*-
# Cell 4：六模型完整指标表（增强后，阈值0.5，含95%CI）
def bootstrap_ci(y_true, y_proba, metric, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        yt, yp = y_true[idx], y_proba[idx]
        if metric == "recall":
            vals.append(recall_score(yt, (yp>=0.5).astype(int), zero_division=0))
        elif metric == "auc":
            vals.append(roc_auc_score(yt, yp))
    return np.percentile(vals, [2.5, 97.5])

rows = []
for name in models:
    yp = proba_after[name]
    ypred = (yp >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, ypred).ravel()
    rec_ci = bootstrap_ci(y_test, yp, "recall")
    auc_ci = bootstrap_ci(y_test, yp, "auc")
    rows.append({
        "模型": name,
        "准确率": round(accuracy_score(y_test, ypred), 4),
        "精确率": round(precision_score(y_test, ypred, zero_division=0), 4),
        "召回率": round(recall_score(y_test, ypred, zero_division=0), 4),
        "召回率95%CI": f"[{rec_ci[0]:.3f}, {rec_ci[1]:.3f}]",
        "特异度": round(tn/(tn+fp), 4),
        "F1": round(f1_score(y_test, ypred, zero_division=0), 4),
        "NPV": round(tn/(tn+fn), 4),
        "AUC-ROC": round(roc_auc_score(y_test, yp), 4),
        "AUC95%CI": f"[{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]",
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    })
df_full = pd.DataFrame(rows)
df_full.to_csv(os.path.join(OUT, "六模型完整指标表.csv"), index=False, encoding="utf-8-sig")
print(df_full[["模型","召回率","召回率95%CI","AUC-ROC","AUC95%CI","特异度","精确率"]].to_string(index=False))

        模型    召回率       召回率95%CI  AUC-ROC       AUC95%CI    特异度    精确率
      逻辑回归 0.9810 [0.958, 1.000]   0.8459 [0.812, 0.879] 0.2372 0.2103
       决策树 0.8987 [0.852, 0.947]   0.7905 [0.755, 0.823] 0.5570 0.2958
      随机森林 0.9177 [0.872, 0.958]   0.8316 [0.798, 0.864] 0.5203 0.2838
ExtraTrees 0.9620 [0.929, 0.993]   0.8354 [0.802, 0.867] 0.4050 0.2508
   XGBoost 0.8354 [0.776, 0.890]   0.8518 [0.818, 0.883] 0.6815 0.3520
  LightGBM 0.8101 [0.745, 0.867]   0.8446 [0.808, 0.876] 0.7012 0.3596


In [5]:
# -*- coding: utf-8 -*-
# Cell 5：召回率柱状图（无标题）+ ROC曲线（不同线型标记）
# ---- 召回率柱状图 ----
recs = {name: recall_score(y_test, (proba_after[name]>=0.5).astype(int), zero_division=0)
        for name in models}
colors = ["#d62728","#1f77b4","#2ca02c","#9467bd","#ff7f0e","#8c564b"]
plt.figure(figsize=(10, 6))
bars = plt.bar(list(recs.keys()), list(recs.values()), color=colors, edgecolor="black", linewidth=1)
for b, v in zip(bars, recs.values()):
    plt.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=11)
plt.ylim(0, 1.05)
plt.ylabel("召回率", fontsize=13)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "召回率柱状图.png"), dpi=300, bbox_inches="tight")
plt.close()
print("召回率柱状图已保存。")

# ---- ROC曲线 ----
configs = [
    ("逻辑回归",  "#000000", "-",  "o", 2.2),
    ("决策树",    "#d62728", "--", "s", 1.8),
    ("随机森林",  "#1f77b4", "-.", "^", 1.8),
    ("ExtraTrees","#2ca02c", ":",  "D", 1.8),
    ("XGBoost",   "#9467bd", (0,(5,5)), "p", 1.8),
    ("LightGBM",  "#ff7f0e", (0,(3,1)), "v", 1.8),
]
plt.figure(figsize=(8, 8))
for name, color, ls, mk, lw in configs:
    fpr, tpr, _ = roc_curve(y_test, proba_after[name])
    plt.plot(fpr, tpr, color=color, linestyle=ls, marker=mk, lw=lw, ms=5,
             mew=1.2, mfc="white", markevery=0.12,
             label=f"{name} (AUC={auc(fpr,tpr):.3f})")
plt.plot([0,1],[0,1], color="gray", linestyle="--", lw=1.2, label="随机分类器")
plt.xlabel("假阳性率(FPR)", fontsize=14)
plt.ylabel("真阳性率(TPR)", fontsize=14)
plt.xlim(-0.01, 1.01); plt.ylim(-0.01, 1.01)
plt.grid(color="lightgray", linestyle="--", linewidth=0.5, alpha=0.5)
plt.legend(fontsize=10.5, loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "ROC曲线.png"), dpi=300, bbox_inches="tight")
plt.close()
print("ROC曲线已保存。")

召回率柱状图已保存。
ROC曲线已保存。


In [6]:
# -*- coding: utf-8 -*-
# Cell 6：DeLong检验（逻辑回归 vs XGBoost / LightGBM）
def delong_test(y_true, pred1, pred2):
    n = len(y_true); n_pos = int(y_true.sum()); n_neg = n - n_pos
    def structure(pred):
        S = np.zeros(n)
        for i in range(n):
            if y_true[i] == 1:
                S[i] = np.sum(pred[y_true==0] < pred[i]) / n_neg
            else:
                S[i] = np.sum(pred[y_true==1] > pred[i]) / n_pos
        return S
    S1, S2 = structure(pred1), structure(pred2)
    auc1, auc2 = roc_auc_score(y_true, pred1), roc_auc_score(y_true, pred2)
    v1_pos = np.var(S1[y_true==1])/n_pos; v1_neg = np.var(S1[y_true==0])/n_neg
    v2_pos = np.var(S2[y_true==1])/n_pos; v2_neg = np.var(S2[y_true==0])/n_neg
    cov_pos = np.cov(S1[y_true==1], S2[y_true==1])[0,1]/n_pos
    cov_neg = np.cov(S1[y_true==0], S2[y_true==0])[0,1]/n_neg
    var_diff = (v1_pos+v1_neg)+(v2_pos+v2_neg)-2*(cov_pos+cov_neg)
    if var_diff <= 0:
        return 0.0, 1.0
    z = (auc1-auc2)/np.sqrt(var_diff)
    return float(z), float(2*(1-norm.cdf(abs(z))))

p_lr = proba_after["逻辑回归"]
rows = []
for nm in ["XGBoost", "LightGBM"]:
    z, pv = delong_test(y_test, p_lr, proba_after[nm])
    rows.append({"对比": f"逻辑回归 vs {nm}", "Z": round(z,4), "P": round(pv,4)})
    print(f"逻辑回归 vs {nm}: Z={z:.4f}, P={pv:.4f}")
pd.DataFrame(rows).to_csv(os.path.join(OUT, "DeLong检验结果.csv"), index=False, encoding="utf-8-sig")

逻辑回归 vs XGBoost: Z=-0.8184, P=0.4131
逻辑回归 vs LightGBM: Z=0.1710, P=0.8642


In [ ]:
# -*- coding: utf-8 -*-
# Cell 7：SHAP分析（蜂群图前10 + 样本226瀑布图前5）
def zh(c):
    for a, b in [("性别_男","性别"), ("民族_汉族","民族(汉)"),
                 ("民族_少数民族","民族(少)"), ("是否独生_是","独生")]:
        c = c.replace(a, b)
    c = c.replace("生源地_","").replace("_","")
    return c
fz = [zh(c) for c in X_train.columns]

# 逻辑回归 SHAP（基于训练集背景）
lr_final = models["逻辑回归"].fit(X_bal, y_bal)  # 确保是最优逻辑回归
scaler = lr_final.named_steps["scaler"]
X_train_shap = pd.DataFrame(scaler.transform(X_train), columns=fz)
X_test_shap  = pd.DataFrame(scaler.transform(X_test), columns=fz)
explainer = shap.LinearExplainer(lr_final.named_steps["model"], X_train_shap)
sv = explainer(X_test_shap)

# ---- 蜂群图：前10特征，无标题，负号修复 ----
plt.figure(figsize=(10, 6))
shap.summary_plot(sv.values, X_test_shap, feature_names=fz, show=False, max_display=10)
plt.title("")
ax = plt.gca()
locs = ax.get_xticks()
ax.set_xticklabels([f"{loc:.1f}" for loc in locs])
ax.set_xlabel("SHAP值", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "SHAP蜂群图.png"), dpi=300, bbox_inches="tight")
plt.close()
print("SHAP蜂群图已保存。")

# ---- 瀑布图：样本226，前5特征+其余，无标题 ----
idx = 226 - 1
p226 = lr_final.predict_proba(X_test.iloc[[idx]])[:,1][0]
plt.figure(figsize=(9, 6))
shap.plots.waterfall(sv[idx], max_display=6, show=False)
plt.title("")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "SHAP瀑布图_样本226.png"), dpi=300, bbox_inches="tight")
plt.close()
print(f"SHAP瀑布图已保存（样本226，预测概率{p226:.4f}）。")

# 保存SHAP值明细
shap_df = pd.DataFrame(sv.values, columns=fz)
shap_df.insert(0, "样本号", range(1, len(shap_df)+1))
shap_df.to_csv(os.path.join(OUT, "SHAP值明细.csv"), index=False, encoding="utf-8-sig")
print("SHAP值明细已保存。")
print("\n全部流程完成。")